Executando de modo manual pra entender como funciona o Langgraph por traś

In [ ]:
import os
import sys

# Garante que o Python encontre a raiz do projeto (onde está o retrieval.py)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.append(project_root)

# Muda o diretório atual de execução para a raiz do projeto
# Assim, caminhos relativos como 'data/processed/...' voltam a funcionar normalmente
os.chdir(project_root)

print(f"📍 Diretório de execução atual: {os.getcwd()}")

In [116]:
from src.retrieval import busca, model, embeddings, chunks

In [117]:
def fazer_busca(query: str, k: int = 5) -> str:
    # utilizando classe criada no retrieval
    buscador = busca(
        model= model,
        vetor= embeddings,
        k= k,
        xq= query
    )

    indices = buscador.metrica(tipo= 1) #IndexFlatL2

    if indices is None or len(indices[0]) == 0:
        return "Nenhuma informação encontrada"
     
    chunks_encontrados = [ # Recupera os chunks de texto originais a partir dos índices retornados pelo FAISS
        chunks[i]['text'] for i in indices[0] if i < len(chunks)
    ]
    
    return "\n\n---\n\n".join(chunks_encontrados)


In [118]:
# Iniciando o modelo
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model = 'gemini-3.5-flash-lite',
    model_provider = 'google_genai'
)

In [119]:
from langchain_core.prompts import ChatPromptTemplate

def traduzir_query_ingles(query: str) -> str:
    ''' Receber a query em pt e traduzir pra ingles'''

    if isinstance(query, dict):
        query = query.get("query", "")

    prompt = ChatPromptTemplate.from_template(
        "Você é um assistente especialista em busca vetorial. "
        "Traduza a seguinte pergunta do usuário para o inglês de forma clara e direta. "
        "Retorne APENAS o texto traduzido em inglês, sem explicações adicionais.\n\n"
        "Pergunta: {query}"
    )

    chain = prompt | llm # pegar a saida do objeto a esquerda e passa como entrada para a direita
    english_query = chain.invoke( # retorna um obj contendo a messagem da IA
        {"query": query}
    )
    
    print(f"🔍 [DEBUG Tool Gemini] Query original: '{query}' | Traduzida: '{english_query}'")

    # Extrai o texto do content (content extrai apenas o texto bruto contido dentro da msg)
    if isinstance(english_query.content, list):
        texto = "".join([
            p if isinstance(p, str) else p.get("text", "") for p in english_query.content
        ])

    else:
        texto = str(english_query.content)

    return texto

In [120]:
from langchain.tools import (
    tool,
    BaseTool
)

In [121]:
@tool
def procurar_repositorio_doc(query) -> str:
    ''' Use when the question is about rules, PEPs, conventions, or official style guides for Python code.
    
    Args:
        query: The user's question, topic, or search terms regarding Python code style/guidelines.

    Returns:
        Relevant excerpts found in the official document repository.
    '''

    query_ingles = traduzir_query_ingles(query)
    return fazer_busca(
        query_ingles, 
        k=5
    )


In [122]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    BaseMessage,
    ToolMessage
)

from langchain_core.tools import BaseTool

In [123]:
# Definir o que e a forma que queremos de resposta. Não é exibido ao ausuário.

system_message = SystemMessage( #Prompt programador
    "Você é um guia de estudos que ajuda programadores, cientistas de dados e afins.\n\n"
    "Evite conversar sobre assuntos paralelos ao tópico escolhido. \n\n"
    "Você pode ser amigável e tratar o estudante conforme ele te tratar. Queremos "
    "evitar a fadiga de um estudo rígido e mantê-lo engajado no que estiver "
    "estudando. Talvez até adicionando algum curiosidade. \n\n"
    "As próximas mensagens serão de um estudante."
)

In [110]:
message_human = str(input("Qual a msg?"))

In [111]:
# Criar historico de mensages
messages: list[BaseMessage] = [
    system_message, message_human
]

In [112]:
# criar a lista de ferramentas
tools: list[BaseTool] = [procurar_repositorio_doc]

# complementar o llm com as tools
llm_tools = llm.bind_tools(tools)

In [113]:
# enviar msg para o modelo
llm_response = llm_tools.invoke(messages)

# Add ao historico
messages.append(llm_response)

llm_response.content é a resposta pela IA  
llm_response.tool_calls (name, arg, id) resposta em dicionario atraves da tool

In [114]:
# verificar se o modelo optou por chamar a ferramentea
if llm_response.tool_calls:
    print('FERRAMENTA ACIONADA. \nO modelo está olhando a doc do repositorio')

    # executar cada tool solicitada pelo modelo
    for tool_call in llm_response.tool_calls:

        tool_name = tool_call['name'] # nome da fonte
        tool_arg = tool_call['args'] # query

        if tool_name == 'procurar_repositorio_doc':
            resultado = procurar_repositorio_doc.invoke(tool_arg) # fazer a buscar no repositorio

            tool_message = ToolMessage( # chamar msg da tool
                content = resultado, 
                tool_call_id = tool_call['id']
            ) # gerar um dicionario de resposta:id da query
            messages.append(tool_message)

    # voltar a segunda chamada pra IA juntar a resposta final com o contexto de busca

    reposta_final = llm_tools.invoke(messages)
    print('Resposta Final (com RAG):')
    print(80*'-')
    print(reposta_final.content)

else:
    print('Resposta Final:')
    print(80*'-')
    print(llm_response.content)
    



Resposta Final:
--------------------------------------------------------------------------------
[{'type': 'text', 'text': 'Fala aí! Tudo bem? \n\nParece que a letra "l" escapou antes da hora. O que manda? Qual vai ser o nosso tópico de estudo ou dúvida de hoje?', 'extras': {'signature': 'El4KXAERTTIPhqjn/77SzG/YWuF4Aem5U7Nr9SIKWfV8rSGoLafx2QeHc/P2uqbcO5XZEh6xVEuE5CjcBDoiFimHG7d8fN7AdBUQX5m9R98InOKGaoIxqn6q1yWi4K/8'}}]


In [115]:
import this